# Data collection for reviwer_ml


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import ee, os, time, json, pandas as pd
from google.colab import drive

try:
    ee.Initialize(project='xx-xxxx-xxxx')
except:
    ee.Authenticate()
    ee.Initialize(project='xx-xxxx-xxxx')

In [ ]:
! pip install rioxarray

In [19]:
"""
================================================================================
PART A: GEE EXTRACTION ONLY (Nighttime Lights & PM2.5) - FINAL CORRECTED
================================================================================
This script is self-contained for Google Colab. It mounts your Drive,
authenticates Earth Engine, and queues 40 export tasks for the reviewers'
requested datasets.

Updates:
- NTL: Handles DMSP/VIIRS era harmonization.
- PM2.5: Uses updated V6 catalog ('GLOBAL-SATELLITE-PM25/ANNUAL') and 0.1 scale factor.
"""

import ee
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Authenticate Earth Engine
try:
    ee.Initialize(project='ee-faiz2009cu')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-faiz2009cu')

# ----------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------
# Note: GEE's toDrive export ignores subfolder slashes and creates a single
# flattened folder at the root of your Google Drive.
DRIVE_EXPORT_FOLDER = "Agricultural_RS_LE_2025 review raw_gee_exports"

EXCLUDE_STATEFP = ["02", "15", "60", "66", "69", "72", "78"]
PCTS = [10, 25, 50, 75, 90]
YEARS = list(range(2000, 2020))

# ----------------------------------------------------------------------------
# HELPER FUNCTIONS
# ----------------------------------------------------------------------------
def _ee_county_fc():
    """CONUS counties from TIGER/2018, matching your extraction filter."""
    counties = ee.FeatureCollection("TIGER/2018/Counties")
    exclude = ee.List(EXCLUDE_STATEFP)
    counties = counties.filter(ee.Filter.inList("STATEFP", exclude).Not())
    def _tag(f):
        fips = ee.String(f.get("STATEFP")).cat(ee.String(f.get("COUNTYFP")))
        return f.set({"FIPS": fips})
    return counties.map(_tag)

def _ee_spatial_reducer():
    """mean + stdDev + percentile[10,25,50,75,90]."""
    r = (ee.Reducer.mean()
         .combine(ee.Reducer.stdDev(), sharedInputs=True)
         .combine(ee.Reducer.percentile(PCTS), sharedInputs=True))
    return r

def _ee_reduce_year(image, counties, scale, year, band_out):
    """reduceRegions with tileScale=4, tag year, return FeatureCollection rows."""
    fc = image.reduceRegions(collection=counties,
                             reducer=_ee_spatial_reducer(),
                             scale=scale, tileScale=4)
    def _tag(f):
        return f.set({"year": year}).setGeometry(None)
    keep = ["FIPS", "STATEFP", "COUNTYFP", "NAME", "year",
            "mean", "stdDev", "p10", "p25", "p50", "p75", "p90"]
    return fc.map(_tag).select(keep, retainGeometry=False)

# ----------------------------------------------------------------------------
# EXTRACTION FUNCTIONS
# ----------------------------------------------------------------------------
def extract_nightlights():
    counties = _ee_county_fc()
    dmsp = ee.ImageCollection("NOAA/DMSP-OLS/NIGHTTIME_LIGHTS")
    viirs = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V21")

    for year in YEARS:
        if year <= 2013:
            img = (dmsp.filterDate(f"{year}-01-01", f"{year}-12-31")
                   .select("stable_lights").mean().rename("ntl"))
            sensor, scale = "DMSP", 1000
        else:
            img = (viirs.filterDate(f"{year}-01-01", f"{year}-12-31")
                   .select("average").mean().rename("ntl"))
            sensor, scale = "VIIRS", 500

        rows = _ee_reduce_year(img, counties, scale, year, "ntl")
        rows = rows.map(lambda f: f.set({"sensor": sensor}))

        task = ee.batch.Export.table.toDrive(
            collection=rows,
            description=f"ntl_raw_{year}",
            folder=DRIVE_EXPORT_FOLDER,
            fileNamePrefix=f"ntl_raw_{year}",
            fileFormat="CSV")
        task.start()
        print(f"[GEE] Queued NTL export {year} ({sensor})")

def extract_pm25():
    counties = _ee_county_fc()

    # Updated to the new V6 path on the sat-io catalog
    pm = ee.ImageCollection("projects/sat-io/open-datasets/GLOBAL-SATELLITE-PM25/ANNUAL")
    BAND = "b1"

    for year in YEARS:
        img = pm.filterDate(f"{year}-01-01", f"{year}-12-31").first()
        img = ee.Image(ee.Algorithms.If(
            img, img,
            pm.filter(ee.Filter.eq("year", year)).first()))

        # The V6 dataset requires scaling the data by 0.1
        img = ee.Image(img).select([BAND], ["pm25"]).multiply(0.1)

        rows = _ee_reduce_year(img, counties, 1000, year, "pm25")
        task = ee.batch.Export.table.toDrive(
            collection=rows,
            description=f"pm25_raw_{year}",
            folder=DRIVE_EXPORT_FOLDER,
            fileNamePrefix=f"pm25_raw_{year}",
            fileFormat="CSV")
        task.start()
        print(f"[GEE] Queued PM2.5 export {year} (V6 Fixed)")

# ----------------------------------------------------------------------------
# EXECUTION
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    print("Submitting tasks to Earth Engine...\n")
    extract_nightlights()
    print("-" * 50)
    extract_pm25()
    print("\nAll 40 tasks successfully queued! Check code.earthengine.google.com/tasks to monitor progress.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Submitting tasks to Earth Engine...

[GEE] Queued NTL export 2000 (DMSP)
[GEE] Queued NTL export 2001 (DMSP)
[GEE] Queued NTL export 2002 (DMSP)
[GEE] Queued NTL export 2003 (DMSP)
[GEE] Queued NTL export 2004 (DMSP)
[GEE] Queued NTL export 2005 (DMSP)
[GEE] Queued NTL export 2006 (DMSP)
[GEE] Queued NTL export 2007 (DMSP)
[GEE] Queued NTL export 2008 (DMSP)
[GEE] Queued NTL export 2009 (DMSP)
[GEE] Queued NTL export 2010 (DMSP)
[GEE] Queued NTL export 2011 (DMSP)
[GEE] Queued NTL export 2012 (DMSP)
[GEE] Queued NTL export 2013 (DMSP)
[GEE] Queued NTL export 2014 (VIIRS)
[GEE] Queued NTL export 2015 (VIIRS)
[GEE] Queued NTL export 2016 (VIIRS)
[GEE] Queued NTL export 2017 (VIIRS)
[GEE] Queued NTL export 2018 (VIIRS)
[GEE] Queued NTL export 2019 (VIIRS)
--------------------------------------------------
[GEE] Queued PM2.5 export 2000 (V6 Fixed)
[GEE] Queued PM

In [18]:
import pandas as pd
import glob
from pathlib import Path
import numpy as np

# ============================================================================
# 1. SETUP PATHS
# ============================================================================
BASE_DIR = Path("/content/drive/MyDrive/Agricultural_RS_LE_2025/review")
MASTER_FILE = BASE_DIR / "full_clean_engineered_dataset_with_LE.csv"
OUTPUT_FILE = BASE_DIR / "full_clean_engineered_dataset_with_LE_NTL_PM25.csv"

print("Loading original master dataset...")
df_master = pd.read_csv(MASTER_FILE)

# --- BEFORE DIAGNOSTIC ---
print("\n" + "="*50)
print("📊 DIAGNOSTIC: BEFORE MERGE")
print("="*50)
print(f"Shape: {df_master.shape[0]:,} rows x {df_master.shape[1]:,} columns")
print(f"Unique Counties (fips): {df_master['fips'].nunique():,}")

# Ensure master fips is perfectly 5-digit string and year is integer
df_master['fips'] = df_master['fips'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(5)
df_master['year'] = df_master['year'].astype(int)

# ============================================================================
# 2. PREP NIGHTTIME LIGHTS (NTL)
# ============================================================================
print("\nProcessing Nighttime Lights...")

# THE FIX: Wildcard search to find the files in ANY of those weirdly named folders
ntl_files = glob.glob("/content/drive/MyDrive/*raw_gee_exports*/ntl_raw_*.csv")

if not ntl_files:
    raise ValueError("Could not find NTL files anywhere. Are they still processing in Earth Engine?")

print(f"Found {len(ntl_files)} NTL files. Merging...")
df_ntl = pd.concat([pd.read_csv(f) for f in ntl_files], ignore_index=True)

# Standardize FIPS
fcol = 'FIPS' if 'FIPS' in df_ntl.columns else ('GEOID' if 'GEOID' in df_ntl.columns else 'fips')
df_ntl['fips'] = df_ntl[fcol].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(5)
df_ntl['year'] = df_ntl['year'].astype(int)

# Drop duplicates just in case Earth Engine exported the same year twice across different folders
df_ntl = df_ntl.drop_duplicates(subset=['fips', 'year', 'sensor'])

# Harmonize DMSP (0-63) and VIIRS (radiance) to a unified [0,1] scale
stat_cols = ["mean", "stdDev", "p10", "p25", "p50", "p75", "p90"]
for c in stat_cols:
    if c in df_ntl.columns:
        col_harm = f"NTL_harmonized_{c}"
        df_ntl[col_harm] = np.nan
        for sensor in df_ntl['sensor'].dropna().unique():
            m = df_ntl['sensor'] == sensor
            v = pd.to_numeric(df_ntl.loc[m, c], errors='coerce')
            lo, hi = np.nanpercentile(v, 1), np.nanpercentile(v, 99)
            df_ntl.loc[m, col_harm] = np.clip((v - lo) / max(hi - lo, 1e-9), 0, 1)

keep_ntl = ['fips', 'year'] + [c for c in df_ntl.columns if 'NTL_harmonized' in c]
df_ntl_clean = df_ntl[keep_ntl].copy()

# ============================================================================
# 3. PREP PM2.5
# ============================================================================
print("\nProcessing PM2.5...")

# THE FIX: Wildcard search
pm_files = glob.glob("/content/drive/MyDrive/*raw_gee_exports*/pm25_raw_*.csv")

if not pm_files:
    raise ValueError("Could not find PM2.5 files anywhere. Are they still processing in Earth Engine?")

print(f"Found {len(pm_files)} PM2.5 files. Merging...")
df_pm = pd.concat([pd.read_csv(f) for f in pm_files], ignore_index=True)

# Standardize FIPS
fcol = 'FIPS' if 'FIPS' in df_pm.columns else ('GEOID' if 'GEOID' in df_pm.columns else 'fips')
df_pm['fips'] = df_pm[fcol].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(5)
df_pm['year'] = df_pm['year'].astype(int)

# Drop duplicates just in case
df_pm = df_pm.drop_duplicates(subset=['fips', 'year'])

# Rename columns to have PM25_ prefix
pm_rename = {}
for c in stat_cols:
    if c in df_pm.columns:
        pm_rename[c] = f"PM25_{c}"

df_pm_clean = df_pm[['fips', 'year'] + list(pm_rename.keys())].rename(columns=pm_rename)

# ============================================================================
# 4. THE MASTER MERGE
# ============================================================================
print("\nMerging into master dataset...")
df_final = df_master.merge(df_ntl_clean, on=['fips', 'year'], how='left')
df_final = df_final.merge(df_pm_clean, on=['fips', 'year'], how='left')

# Leakage-safe intra-county fill
new_cols = [c for c in df_final.columns if c.startswith(('NTL_', 'PM25_'))]
df_final = df_final.sort_values(["fips", "year"]).reset_index(drop=True)
df_final[new_cols] = df_final.groupby("fips")[new_cols].bfill().ffill()

# --- AFTER DIAGNOSTIC ---
print("\n" + "="*50)
print("📊 DIAGNOSTIC: AFTER MERGE")
print("="*50)
print(f"Shape: {df_final.shape[0]:,} rows x {df_final.shape[1]:,} columns")
print(f"Row count changed? {'⚠️ YES (WARNING)' if df_master.shape[0] != df_final.shape[0] else '✅ NO (PERFECT)'}")

print("\nMissing Values Check (New Features):")
for c in new_cols:
    missing = df_final[c].isna().sum()
    status = "✅ Perfect" if missing == 0 else f"⚠️ {missing} missing"
    print(f" - {c:<25}: {status}")

print("\nSaving final dataset...")
df_final.to_csv(OUTPUT_FILE, index=False)
print(f"🎉 Successfully saved to:\n{OUTPUT_FILE}")

Loading original master dataset...

📊 DIAGNOSTIC: BEFORE MERGE
Shape: 62,540 rows x 464 columns
Unique Counties (fips): 3,127

Processing Nighttime Lights...
Found 20 NTL files. Merging...

Processing PM2.5...
Found 20 PM2.5 files. Merging...

Merging into master dataset...

📊 DIAGNOSTIC: AFTER MERGE
Shape: 62,540 rows x 478 columns
Row count changed? ✅ NO (PERFECT)

Missing Values Check (New Features):
 - NTL_harmonized_mean      : ✅ Perfect
 - NTL_harmonized_stdDev    : ✅ Perfect
 - NTL_harmonized_p10       : ✅ Perfect
 - NTL_harmonized_p25       : ✅ Perfect
 - NTL_harmonized_p50       : ✅ Perfect
 - NTL_harmonized_p75       : ✅ Perfect
 - NTL_harmonized_p90       : ✅ Perfect
 - PM25_mean                : ✅ Perfect
 - PM25_stdDev              : ✅ Perfect
 - PM25_p10                 : ✅ Perfect
 - PM25_p25                 : ✅ Perfect
 - PM25_p50                 : ✅ Perfect
 - PM25_p75                 : ✅ Perfect
 - PM25_p90                 : ✅ Perfect

Saving final dataset...
🎉 Succes

In [20]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Agricultural_RS_LE_2025/review")

print("Saving the standalone NTL and PM2.5 files for the HPC...")

# Save the raw Nighttime Lights dataset
df_ntl.to_csv(BASE_DIR / "ntl_county_year_raw.csv", index=False)
print("✅ Saved ntl_county_year_raw.csv")

# Save the raw PM2.5 dataset
df_pm.to_csv(BASE_DIR / "pm25_county_year.csv", index=False)
print("✅ Saved pm25_county_year.csv")

print("\nDone! You now have all the files ready for the HPC.")

Saving the standalone NTL and PM2.5 files for the HPC...
✅ Saved ntl_county_year_raw.csv
✅ Saved pm25_county_year.csv

Done! You now have all the files ready for the HPC.
